# USD/JPY Trading Model Trainer (Google Colab)

This notebook runs the entire training and feature-pruning pipeline on Google Colab's free GPUs. 

**Pre-requisite:**
MetaTrader5 only runs on Windows. You must first download the raw USD/JPY data on your local machine and zip your `train_pipeline` folder to upload here.

### Step 1: Upload your code and data
1. Zip your local `train_pipeline` folder (make sure it contains your `data/usdjpy_m1.csv` if you downloaded it locally).
2. Run the cell below, which will prompt you to upload the zip file. It will automatically extract it.

In [ ]:
from google.colab import files
import zipfile
import os

# Upload the zip file
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract it
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('.')
    
print(f"Extracted {zip_name} successfully!")

### Step 2: Install Dependencies
Colab already has PyTorch installed with CUDA support. We just need to add a few missing libraries.

In [ ]:
!pip install lightgbm shap pandas numpy scikit-learn numba ta

### Step 3: Run Data Processing (Synthetic Microstructure & Labels)
Change the filenames if your USD/JPY data is named differently.

In [ ]:
!python train_pipeline/synthetic_microstructure.py \
    --data train_pipeline/data/usdjpy_m1.csv \
    --out train_pipeline/data/usdjpy_m1_synmicro.csv

!python train_pipeline/triple_barrier_labels.py \
    --data train_pipeline/data/usdjpy_m1_synmicro.csv \
    --out train_pipeline/data/usdjpy_m1_tb.csv \
    --pt-atr 2.0 --sl-atr 1.0 --max-hold 15

### Step 4: Train the Model (Using Native CUDA GPU)
This will run significantly faster on Colab's T4 GPU.

In [ ]:
!python train_pipeline/train_sota_v2.py \
    --data train_pipeline/data/usdjpy_m1_tb.csv \
    --out-dir train_pipeline/reports/usdjpy \
    --seq-len 60 --patch-len 8 --epochs 40 --gpu

### Step 5: Prune Noise Features using SHAP
Find out which features to drop for USD/JPY.

In [ ]:
!python train_pipeline/prune_features.py \
    --model train_pipeline/reports/usdjpy/patchtst_primary.pt \
    --data train_pipeline/data/usdjpy_m1_tb.csv \
    --samples 200

### Step 6: Download the Trained Model
Run this to download the checkpoint back to your Windows PC so you can trade with it.

In [ ]:
from google.colab import files
files.download('train_pipeline/reports/usdjpy/patchtst_primary.pt')
files.download('train_pipeline/reports/usdjpy/sota_config.json')